# Commercial LLM Evaluation: ChatGPT vs Claude

This notebook mirrors the workflow of `PLS_SFT_Colab_L4_Stable (3).ipynb` but focuses on **evaluating commercial LLM APIs**.
It targets Python 3.12 kernels for the end-to-end evaluation flow. AlignScore runs through a dedicated Python 3.10 virtual environment (mirroring the Colab notebook) so the rest of the runtime stays on modern 3.12 stacks.


## 0) Environment expectations
Create or activate a Python 3.12 environment before running additional cells. Example:
```bash
python3.12 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
```
On Google Colab, choose the **Python 3.12** runtime. The AlignScore section later in this notebook spins up an isolated Python 3.10 virtualenv (same pattern as `PLS_SFT_Colab_L4_Stable (3).ipynb`) so the rest of the notebook remains compatible with Python 3.12.


### Check Python/runtime setup (explained)
Confirms interpreter, OS, and GPU availability so we can branch later if needed.

In [ ]:
import platform
import sys
import torch

print(f"Python: {sys.version}")
if not sys.version.startswith('3.12'):
    print('⚠️ Recommended interpreter is Python 3.12.x (AlignScore uses its own 3.10 venv later).')
print(f"Platform: {platform.platform()}")
print(f"PyTorch: {torch.__version__}")
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


### Install pinned dependencies (explained)
Installs API SDKs and evaluation libraries (BERTScore, readability metrics, AlignScore). Versions are compatible with Python 3.10.

In [ ]:
%%bash
set -euo pipefail
pip install --upgrade   openai==1.37.1   anthropic==1.21.2   python-dotenv==1.0.1   pandas==2.2.2   textstat==0.7.4   evaluate==0.4.2   bert-score==0.3.13   datasets==2.20.0   tqdm==4.66.4   numpy==2.1.3   torch==2.5.1


### Load API keys and shared config (explained)
Reads credentials from `.env`/environment variables so secrets never appear in the notebook output.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
if not OPENAI_API_KEY:
    print('Missing OPENAI_API_KEY in env/.env')
if not ANTHROPIC_API_KEY:
    print('Missing ANTHROPIC_API_KEY in env/.env')

### Central evaluation config (explained)
Capture tunable parameters—dataset paths, prompts, model IDs, token budgets, and safety flags—in a dataclass for clarity.

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


def default_align_env_path() -> Path:
    colab_root = Path('/content')
    if colab_root.exists():
        return colab_root / 'envs' / 'py310-alignscore'
    return Path.home() / '.venvs' / 'py310-alignscore'


@dataclass
class EvalConfig:
    data_path: Path = Path('../data/pls_eval_pairs.csv')
    output_path: Path = Path('../results/commercial_llm_eval.parquet')
    system_prompt: str = (
        'You are a senior scientific writer. Produce a plain-language summary that is factual, concise, and 6th-grade readable.'
    )
    user_template: str = (
        'Summarize the following abstract into 4-5 sentences focusing on outcomes and limitations.

"{article}"'
    )
    openai_model: str = 'gpt-4o-mini'
    anthropic_model: str = 'claude-3-haiku-20240307'
    temperature: float = 0.2
    max_output_tokens: int = 512
    max_requests: Optional[int] = 20
    dry_run: bool = False  # flip to True to inspect the pipeline without hitting APIs
    alignscore_input_path: Path = Path('../results/commercial_llm_alignscore_payload.csv')
    alignscore_output_path: Path = Path('../results/commercial_llm_alignscore_scores.csv')
    alignscore_env_path: Path = field(default_factory=default_align_env_path)


config = EvalConfig()
config


### Load evaluation dataset (explained)
Reads article/reference pairs into a DataFrame and optionally caps the sample via `config.max_requests` for cheap smoke tests.

In [ ]:
import pandas as pd

if not config.data_path.exists():
    raise FileNotFoundError(f'Dataset not found: {config.data_path}')
raw_df = pd.read_csv(config.data_path)
required_cols = {'article', 'reference_summary'}
missing = required_cols.difference(raw_df.columns)
if missing:
    raise ValueError(f'Dataset is missing columns: {missing}')
if config.max_requests:
    raw_df = raw_df.head(config.max_requests)
print(raw_df.head(2))
print(f'Total samples: {len(raw_df)}')

### Build OpenAI & Anthropic clients (explained)
Initializes SDK clients lazily so the notebook can still run analytics-only cells on machines without credentials.

In [ ]:
from typing import Optional
from openai import OpenAI
from anthropic import Anthropic

openai_client: Optional[OpenAI] = None
anthropic_client: Optional[Anthropic] = None

if OPENAI_API_KEY:
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
if ANTHROPIC_API_KEY:
    anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)

print('OpenAI client ready:', bool(openai_client))
print('Anthropic client ready:', bool(anthropic_client))

### Prompt assembly helper (explained)
Ensures each article is formatted consistently before sending it to either provider.

In [ ]:
def build_prompt(article: str) -> str:
    return config.user_template.format(article=article.strip())

raw_df = raw_df.assign(prompt=raw_df['article'].apply(build_prompt))
raw_df.head(2)

### Generation helpers for ChatGPT and Claude (explained)
Wraps provider-specific SDK calls, handles token accounting, and honors `config.dry_run` to avoid accidental billing.

In [ ]:
from time import perf_counter
from typing import Dict, Any

class InferenceError(RuntimeError):
    pass


def _maybe_abort(provider: str):
    if config.dry_run:
        raise InferenceError(f'Dry-run enabled; skipping {provider} call')


def generate_with_chatgpt(prompt: str) -> Dict[str, Any]:
    if openai_client is None:
        raise InferenceError('OpenAI client is not configured')
    _maybe_abort('OpenAI')
    start = perf_counter()
    response = openai_client.responses.create(
        model=config.openai_model,
        input=[
            {"role": "system", "content": config.system_prompt},
            {"role": "user", "content": prompt},
        ],
        temperature=config.temperature,
        max_output_tokens=config.max_output_tokens,
    )
    text_chunks = []
    for item in response.output:
        for content in item.content:
            if content.type == 'output_text':
                text_chunks.append(content.text)
    generation = '
'.join(text_chunks).strip()
    usage = response.usage
    elapsed = perf_counter() - start
    return {
        'generation': generation,
        'prompt_tokens': getattr(usage, 'input_tokens', None),
        'completion_tokens': getattr(usage, 'output_tokens', None),
        'total_tokens': getattr(usage, 'total_tokens', None),
        'latency_seconds': elapsed,
    }


def generate_with_claude(prompt: str) -> Dict[str, Any]:
    if anthropic_client is None:
        raise InferenceError('Anthropic client is not configured')
    _maybe_abort('Anthropic')
    start = perf_counter()
    response = anthropic_client.messages.create(
        model=config.anthropic_model,
        system=config.system_prompt,
        temperature=config.temperature,
        max_tokens=config.max_output_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    text_chunks = [block.text for block in response.content if block.type == 'text']
    generation = '
'.join(text_chunks).strip()
    usage = response.usage
    elapsed = perf_counter() - start
    return {
        'generation': generation,
        'prompt_tokens': getattr(usage, 'input_tokens', None),
        'completion_tokens': getattr(usage, 'output_tokens', None),
        'total_tokens': getattr(usage, 'input_tokens', 0) + getattr(usage, 'output_tokens', 0),
        'latency_seconds': elapsed,
    }

### Batch inference orchestrator (explained)
Loops over the dataset, dispatches to ChatGPT/Claude generators, and stores telemetry for downstream metrics.

In [ ]:
from tqdm import tqdm

records = []

def run_model(df: pd.DataFrame, provider: str) -> None:
    generator = generate_with_chatgpt if provider == 'chatgpt' else generate_with_claude
    for row in tqdm(df.itertuples(index=False), total=len(df), desc=provider):
        try:
            result = generator(row.prompt)
        except InferenceError as err:
            print(f'Skipping sample due to {err}')
            continue
        except Exception as exc:
            print(f'Provider {provider} failed: {exc}')
            continue
        records.append({
            'provider': provider,
            'article': row.article,
            'reference_summary': row.reference_summary,
            'prompt': row.prompt,
            **result,
        })

for provider in ['chatgpt', 'claude']:
    print(f'Running provider: {provider}')
    run_model(raw_df, provider)

results_df = pd.DataFrame(records).reset_index(drop=True)
results_df['row_id'] = results_df.index
results_df.head()


### Metric utilities (explained)
Defines scorers for BERTScore, readability (Flesch/FKGL/Gunning Fog/SMOG/Coleman-Liau), and AlignScore. Falls back to NaN if AlignScore cannot load on the current hardware.

In [ ]:
import numpy as np
import evaluate
from textstat import textstat

_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
bertscore = evaluate.load('bertscore')

def compute_bertscore(df: pd.DataFrame) -> pd.Series:
    payload = bertscore.compute(
        predictions=df['generation'].tolist(),
        references=df['reference_summary'].tolist(),
        model_type='microsoft/deberta-large-mnli',
        lang='en',
        device=_DEVICE,
    )
    return pd.Series(payload['f1'], index=df.index)


def readability_scores(text: str) -> dict:
    return {
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
        'gunning_fog': textstat.gunning_fog(text),
        'smog_index': textstat.smog_index(text),
        'coleman_liau_index': textstat.coleman_liau_index(text),
    }


### Evaluate generations (explained)
Applies every metric to per-provider groups and attaches the results to `results_df`.

In [ ]:
metric_frames = []
for provider, group in results_df.groupby('provider'):
    idx = group.index
    bert = compute_bertscore(group)
    readability = group['generation'].apply(readability_scores).apply(pd.Series)
    metrics = pd.DataFrame({
        'provider': provider,
        'bertscore_f1': bert,
    }, index=idx).join(readability)
    metrics['align_score'] = np.nan
    metric_frames.append(metrics)
metrics_df = pd.concat(metric_frames).sort_index()
results_df = results_df.join(metrics_df)
results_df.head()


### AlignScore (isolated Python 3.10 workflow)
AlignScore depends on Torch 1.13/Python 3.10, so we export the generated samples, score them inside a dedicated virtualenv (same pattern used in `PLS_SFT_Colab_L4_Stable (3).ipynb`), and then merge the scores back. Skip these cells if AlignScore is not required.


In [ ]:
align_payload_cols = ['row_id', 'article', 'generation', 'reference_summary']
align_payload = results_df[align_payload_cols].copy()
align_payload_path = config.alignscore_input_path
align_payload_path.parent.mkdir(parents=True, exist_ok=True)
align_payload.to_csv(align_payload_path, index=False)
print(f'Saved {len(align_payload)} rows to {align_payload_path}')
print(f'Default AlignScore venv: {config.alignscore_env_path}')
print('Set ALIGN_VENV / ALIGNSCORE_INPUT / ALIGNSCORE_OUTPUT to override defaults if needed.')


In [ ]:
%%bash
# Create a fresh Python 3.10 env dedicated to AlignScore (safe for Colab)
set -euo pipefail
ENV_DIR=${ALIGN_VENV:-/content/envs/py310-alignscore}

sudo apt-get update -y >/dev/null
sudo apt-get install -y python3.10 python3.10-distutils python3.10-venv >/dev/null
if ! python3.10 -m ensurepip --upgrade >/dev/null 2>&1; then
  curl -sS https://bootstrap.pypa.io/get-pip.py | sudo python3.10
fi

rm -rf "${ENV_DIR}"
python3.10 -m venv "${ENV_DIR}"
source "${ENV_DIR}/bin/activate"

python -m pip install --upgrade pip wheel setuptools
pip install --no-cache-dir torch==1.13.1
pip install --no-cache-dir numpy==1.26.4 pandas==2.2.2 requests==2.32.3
pip install --no-cache-dir transformers==4.40.1 sentencepiece==0.1.99
pip install --no-cache-dir "git+https://github.com/yuh-zha/AlignScore.git"

python - <<'PY'
import sys, torch
print('Python:', sys.executable)
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
import alignscore
print('AlignScore import OK')
PY


In [ ]:
%%bash
# Run AlignScore inside the dedicated env and write outputs back to CSV
set -euo pipefail
VENV=${ALIGN_VENV:-/content/envs/py310-alignscore}
CSV_IN=${ALIGNSCORE_INPUT:-../results/commercial_llm_alignscore_payload.csv}
CSV_OUT=${ALIGNSCORE_OUTPUT:-../results/commercial_llm_alignscore_scores.csv}

if [ ! -d "${VENV}" ]; then
  echo "AlignScore venv not found at ${VENV}. Run the setup cell first."
  exit 1
fi
if [ ! -f "${CSV_IN}" ]; then
  echo "Payload not found at ${CSV_IN}. Re-run the export cell."
  exit 1
fi

source "${VENV}/bin/activate"
export HF_HUB_ENABLE_HF_TRANSFER=0
export HF_HUB_DISABLE_TELEMETRY=1

python - <<'PY'
import os
from pathlib import Path
import numpy as np
import pandas as pd

csv_in = Path(os.environ.get('ALIGNSCORE_INPUT', '../results/commercial_llm_alignscore_payload.csv'))
csv_out = Path(os.environ.get('ALIGNSCORE_OUTPUT', '../results/commercial_llm_alignscore_scores.csv'))

if not csv_in.exists():
    raise FileNotFoundError(f'AlignScore payload missing: {csv_in}')

df = pd.read_csv(csv_in)
required = {'row_id', 'article', 'generation'}
missing = required.difference(df.columns)
if missing:
    raise RuntimeError(f'Payload missing columns: {missing}')

def _clean(series):
    return series.fillna('').map(lambda t: ' '.join(str(t).split()).strip())

df['_article'] = _clean(df['article'])
df['_generation'] = _clean(df['generation'])
if 'reference_summary' in df.columns:
    df['_reference'] = _clean(df['reference_summary'])

mask = df['_article'].str.len().ge(30) & df['_generation'].str.len().ge(30)
valid = df.loc[mask].copy()

from alignscore import AlignScorer

model_name = os.environ.get('ALIGNSCORE_MODEL', 'roberta-large')
batch_size = int(os.environ.get('ALIGNSCORE_BATCH', '4'))
device = 'cuda' if os.environ.get('CUDA_VISIBLE_DEVICES') else 'cpu'

scores = [np.nan] * len(df)
if not valid.empty:
    scorer = AlignScorer(model_name=model_name, batch_size=batch_size, device=device)
    references = valid['_reference'].tolist() if '_reference' in valid.columns else None
    scored = scorer.score(
        contexts=valid['_article'].tolist(),
        summaries=valid['_generation'].tolist(),
        references=references,
    )
    for idx, value in zip(valid.index, scored):
        scores[idx] = value

out = df[['row_id']].copy()
out['align_score'] = scores
out.to_csv(csv_out, index=False)
print(f'Saved AlignScore scores to {csv_out}')
PY


In [ ]:
align_output = config.alignscore_output_path
if align_output.exists():
    align_df = pd.read_csv(align_output)
    required = {'row_id', 'align_score'}
    if required.issubset(align_df.columns):
        score_map = align_df.drop_duplicates('row_id').set_index('row_id')['align_score']
        results_df['align_score'] = results_df['row_id'].map(score_map)
        print(f'Merged AlignScore values from {align_output}')
    else:
        print(f'AlignScore output missing columns: {align_df.columns}')
else:
    print(f'AlignScore output not found ({align_output}); align_score remains NaN')
results_df[['provider', 'row_id', 'align_score']].head()


### Aggregate + compare providers (explained)
Computes macro averages for latency, token usage, BERTScore, AlignScore, and readability measures. AlignScore remains NaN unless you run the isolated Python 3.10 cells below.


In [ ]:
summary = (
    results_df.groupby('provider')[
        [
            'prompt_tokens',
            'completion_tokens',
            'total_tokens',
            'latency_seconds',
            'bertscore_f1',
            'align_score',
            'flesch_reading_ease',
            'flesch_kincaid_grade',
            'gunning_fog',
            'smog_index',
            'coleman_liau_index',
        ]
    ]
    .mean()
    .rename(columns={'latency_seconds': 'avg_latency_seconds'})
)
summary

### Persist detailed outputs (explained)
Writes the full per-sample table (prompts, generations, metrics) so experiments are reproducible outside this notebook.

In [ ]:
config.output_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_parquet(config.output_path, index=False)
print(f'Wrote detailed results to {config.output_path}')